# Module 9: Transformation Operations

Run in Google Colab or a local PySpark environment.

In [ ]:
# Run this cell first
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("LearningModules").getOrCreate()
sc = spark.sparkContext

In [ ]:
# Load datasets (adjust path if needed)
students_df = spark.read.csv("../data/students.csv", header=True, inferSchema=True)
enrollments_df = spark.read.csv("../data/enrollments.csv", header=True, inferSchema=True)

## Joins

Joins combine rows from two DataFrames based on a related column.

In [ ]:
# Inner join: only matching rows
inner_joined = students_df.join(enrollments_df, on="student_id", how="inner")
print(f"Inner join rows: {inner_joined.count()}")
inner_joined.show(5)

In [ ]:
# Left join: all students, even without enrollments
left_joined = students_df.join(enrollments_df, on="student_id", how="left")
print(f"Left join rows: {left_joined.count()}")
left_joined.show(5)

In [ ]:
# Anti join: students with NO enrollments
not_enrolled = students_df.join(enrollments_df, on="student_id", how="anti")
print(f"Students not enrolled in any course: {not_enrolled.count()}")
not_enrolled.show()

## groupBy with Multiple Aggregations

In [ ]:
# Multiple aggregation functions
enrollments_df.groupBy("course_id").agg(
    count("student_id").alias("num_students"),
    countDistinct("student_id").alias("unique_students")
).show()

In [ ]:
# Aggregation on joined data
inner_joined.groupBy("course_id").agg(
    count("student_id").alias("enrollment_count"),
    avg("gpa").alias("avg_gpa"),
    max("gpa").alias("max_gpa"),
    min("gpa").alias("min_gpa")
).orderBy(desc("avg_gpa")).show()

## Pivot

Pivot transforms rows into columns, useful for creating summary tables.

In [ ]:
# Add GPA category for pivoting
with_category = inner_joined.withColumn(
    "gpa_level",
    when(col("gpa") >= 3.5, "High")
    .when(col("gpa") >= 3.0, "Medium")
    .otherwise("Low")
)

# Pivot: count of students per GPA level per course
pivoted = with_category.groupBy("course_id").pivot("gpa_level").count()
pivoted.show()

## Window Functions

Window functions compute values across a set of rows related to the current row.

In [ ]:
from pyspark.sql.window import Window

# Rank students by GPA
window_spec = Window.orderBy(desc("gpa"))

ranked = students_df.withColumn("rank", rank().over(window_spec))
ranked.show()

In [ ]:
# Lag: compare each student's GPA with the previous student
with_lag = students_df.withColumn(
    "prev_gpa", lag("gpa", 1).over(window_spec)
).withColumn(
    "gpa_diff", col("gpa") - col("prev_gpa")
)
with_lag.show()

## Practice Problem 1: Left Join and Filter

Perform a left join of students with enrollments and find students who have null course_id (not enrolled in anything).

<details><summary>Hint</summary>Left join on student_id, then filter where course_id is null.</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
left_joined = students_df.join(enrollments_df, on="student_id", how="left")
not_enrolled = left_joined.filter(col("course_id").isNull())
not_enrolled.show()

## Practice Problem 2: Pivot Table

Create a pivot table showing the average GPA per course_id, pivoted by a gpa_level column (High >= 3.5, Low < 3.5).

<details><summary>Hint</summary>Add gpa_level with when/otherwise, then groupBy("course_id").pivot("gpa_level").agg(avg("gpa")).</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
with_level = inner_joined.withColumn(
    "gpa_level",
    when(col("gpa") >= 3.5, "High").otherwise("Low")
)

pivot_avg = with_level.groupBy("course_id").pivot("gpa_level").agg(avg("gpa"))
pivot_avg.show()

## Practice Problem 3: Window Rank per Course

Rank students within each course by GPA (highest first) using a window function partitioned by course_id.

<details><summary>Hint</summary>Use Window.partitionBy("course_id").orderBy(desc("gpa")), then rank().over(window).</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
course_window = Window.partitionBy("course_id").orderBy(desc("gpa"))

ranked_per_course = inner_joined.withColumn(
    "rank_in_course", rank().over(course_window)
)
ranked_per_course.select("course_id", "name", "gpa", "rank_in_course").show()

## Practice Problem 4: Running Total with Window

Compute a running total of enrollments ordered by student_id using a window function.

<details><summary>Hint</summary>Use Window.orderBy("student_id").rowsBetween(Window.unboundedPreceding, Window.currentRow) with sum(lit(1)).</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
running_window = Window.orderBy("student_id").rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

with_running_total = enrollments_df.withColumn(
    "running_total", sum(lit(1)).over(running_window)
)
with_running_total.show()